# Day 7: Channel-Based Sharding
**Situation:** We change the logic so that 'Messages go where the channel lives'
**Problem:** One channel becomes viral, and all traffic goes to one shard.

In [ ]:
class Message:
    def __init__(self, user_id, channel_id, content):
        self.user_id = user_id
        self.channel_id = channel_id
        self.content = content

class Shard:
    def __init__(self, shard_id):
        self.id = shard_id
        self.messages = []

    def store(self, message):
        self.messages.append(message)

class ShardManager:
    def __init__(self, num_shards):
        self.shards = [Shard(i) for i in range(num_shards)]

    def stats(self):
        for shard in self.shards:
            print(f"Shard {shard.id}: {len(shard.messages)} messages")

In [ ]:
class ChannelShardManager(ShardManager):

    def get_shard(self, channel_id):
        return self.shards[channel_id % len(self.shards)]

    def send_message(self, message):
        shard = self.get_shard(message.channel_id)
        shard.store(message)

### Simulation: A Viral Channel
Let's simulate random traffic across many channels, but one channel (Channel 99) gets a huge spike because it goes viral.

In [ ]:
manager = ChannelShardManager(3)

# Normal Traffic across 50 channels
import random
for _ in range(1000):
        # 100 users, 50 channels
        msg = Message(user_id=random.randint(1, 100), channel_id=random.randint(1, 50), content="Normal message")
        manager.send_message(msg)

# Viral Event on Channel 99
for _ in range(5000):
        msg = Message(user_id=random.randint(1, 100), channel_id=99, content="Omg did you see that?!")
        manager.send_message(msg)

manager.stats()

### Comparison with User-Based Sharding (Day 6)
- **User-Based:** Fails when one user sends too many messages. However, reading a whole channel's history means we have to query *all* shards because users in the same channel are on different shards.
- **Channel-Based:** Helps keep channel data together (fetching history is easy: just query one shard). But it fails terribly when one channel goes viral (a 'Hotspot'), overloading a single shard while others sit idle.